# MuseTalk 1.5 on Google Colab（文本生成视频 口型对应）

运行时请在 **Runtime > Change runtime type** 中选择 **T4 GPU**（或其他 NVIDIA GPU）。按顺序运行下面的单元格，最后一个单元格会输出 Gradio 公网链接。

模型权重约占数 GB，首次启动需要几分钟；Colab 运行时重置后需要重新下载。

In [43]:
# 1) 检查 Colab GPU，并准备项目目录
import os, subprocess, sys

PROJECT_DIR = '/content/MuseTalk'
PROJECT_REPO = os.environ.get('MUSETALK_REPO', 'https://github.com/smclw/MuseTalk.git')

if os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    # Reuse the folder but refresh code when a Colab runtime is rerun.
    subprocess.run(['git', '-C', PROJECT_DIR, 'remote', 'set-url', 'origin', PROJECT_REPO], check=True)
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', '--depth', '1', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', PROJECT_DIR, 'reset', '--hard', 'FETCH_HEAD'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
print('Project:', os.getcwd())
subprocess.run(['nvidia-smi'], check=False)


Project: /content/MuseTalk


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [ ]:
# Colab currently uses Python 3.13 and ships a CUDA-enabled PyTorch build.
# Use subprocess(check=True) so a failed dependency install stops the notebook.
import subprocess, sys
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'], check=True)

def pip_install(*packages, no_deps=False):
    command = [sys.executable, '-m', 'pip', 'install', '-q']
    if no_deps:
        command.append('--no-deps')
    command.extend(packages)
    subprocess.run(command, check=True)

if sys.version_info >= (3, 13):
    pip_install('setuptools>=75.8.0')
    # Python 3.13 has no wheels for the old torch/numpy pins or chumpy.
    pip_install('diffusers==0.30.2', 'accelerate==0.28.0', 'peft==0.11.1', 'numpy>=2.1,<3',
                'opencv-python>=4.10', 'soundfile==0.12.1', 'transformers==4.39.2',
                'huggingface_hub==0.30.2', 'librosa==0.11.0', 'einops==0.8.1',
                'gradio==5.24.0', 'gdown', 'requests>=2.32', 'imageio==2.37.4',
                'imageio-ffmpeg==0.6.0', 'omegaconf', 'ffmpeg-python', 'moviepy==1.0.3')
    pip_install('mmengine==0.10.7', 'mmcv-lite==2.1.0', 'mmdet==3.3.0')
    pip_install('json-tricks', 'munkres', 'scipy', 'matplotlib', 'pycocotools', 'yapf')
    pip_install('mmpose==1.3.2', no_deps=True)
else:
    pip_install('torch==2.0.1', 'torchvision==0.15.2', 'torchaudio==2.0.2',
                '--index-url', 'https://download.pytorch.org/whl/cu118')
    pip_install('-r', 'requirements-colab.txt')
    pip_install('mmengine==0.10.7', 'mmcv==2.0.1', 'mmdet==3.1.0', 'mmpose==1.1.0')


In [ ]:
# 3) 下载 MuseTalk 1.5 所需模型（已存在的文件会跳过）
from huggingface_hub import hf_hub_download
from pathlib import Path

MODELS = Path('models')
FILES = [
    ('TMElyralab/MuseTalk', 'musetalkV15/musetalk.json', MODELS / 'musetalkV15' / 'musetalk.json'),
    ('TMElyralab/MuseTalk', 'musetalkV15/unet.pth', MODELS / 'musetalkV15' / 'unet.pth'),
    ('stabilityai/sd-vae-ft-mse', 'config.json', MODELS / 'sd-vae' / 'config.json'),
    ('stabilityai/sd-vae-ft-mse', 'diffusion_pytorch_model.bin', MODELS / 'sd-vae' / 'diffusion_pytorch_model.bin'),
    ('openai/whisper-tiny', 'config.json', MODELS / 'whisper' / 'config.json'),
    ('openai/whisper-tiny', 'pytorch_model.bin', MODELS / 'whisper' / 'pytorch_model.bin'),
    ('openai/whisper-tiny', 'preprocessor_config.json', MODELS / 'whisper' / 'preprocessor_config.json'),
    ('yzd-v/DWPose', 'dw-ll_ucoco_384.pth', MODELS / 'dwpose' / 'dw-ll_ucoco_384.pth'),
    ('ByteDance/LatentSync', 'latentsync_syncnet.pt', MODELS / 'syncnet' / 'latentsync_syncnet.pt'),
    ('ManyOtherFunctions/face-parse-bisent', '79999_iter.pth', MODELS / 'face-parse-bisent' / '79999_iter.pth'),
    ('ManyOtherFunctions/face-parse-bisent', 'resnet18-5c106cde.pth', MODELS / 'face-parse-bisent' / 'resnet18-5c106cde.pth'),
]
for repo_id, filename, target in FILES:
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.stat().st_size > 0:
        print(f'[skip] {target}')
        continue
    print(f'[download] {repo_id}/{filename}')
    # Keep repository subdirectories (for example musetalkV15/...) under models/.
    download_root = MODELS if '/' in filename else target.parent
    downloaded = hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(download_root))
    if Path(downloaded) != target:
        Path(downloaded).replace(target)

required = [target for _, _, target in FILES]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Missing model files: ' + ', '.join(missing))
print(f'{len(required)} model files are ready.')


In [ ]:
# 4) 核心依赖与 CUDA 自检
import torch, gradio, mmcv, mmpose
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('未检测到 GPU。请在 Colab 中启用 NVIDIA GPU runtime。')
print('GPU:', torch.cuda.get_device_name(0))
print('gradio:', gradio.__version__, 'mmcv:', mmcv.__version__, 'mmpose:', mmpose.__version__)
!ffmpeg -version | head -n 1


In [44]:
# 5) 启动 MuseTalk WebUI
# --share 会生成可从浏览器访问的临时 Gradio 公网链接。
!python app.py --use_float16 --share --ip 0.0.0.0 --ffmpeg_path /usr/bin

All required model files exist.
Loads checkpoint by local backend from path: ./models/dwpose/dw-ll_ucoco_384.pth
cuda start
Downloading: "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth" to /root/.cache/torch/hub/checkpoints/s3fd-619a316812.pth
100% 85.7M/85.7M [00:05<00:00, 16.0MB/s]
An error occurred while trying to fetch models/sd-vae: Error no file named diffusion_pytorch_model.safetensors found in directory models/sd-vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
load unet model from ./models/musetalkV15/unet.pth
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://3bd5ed17834f576fa3.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
reading images...
100% 1/1 [00:00<00:00, 30.76steps/s]
get key_landmark and face boundin